# 12. Тестирование pipeline детекции и идентификации

Notebook демонстрирует:
1. Построение detection response (bbox, mask, species)
2. Background removal (rembg)
3. Работу с config.yaml (species mapping)
4. Полный ViT-pipeline для идентификации особей

In [ ]:
import io
import random
import base64
from PIL import Image

# Укажи путь к файлу
filename = "000fe6ebfc9893.jpg"
filepath = "../../data/datasets/" + filename

# Читаем байты изображения
with open(filepath, "rb") as f:
    img_bytes = f.read()

# Преобразуем изображение
img = Image.open(io.BytesIO(img_bytes)).convert("RGBA")

# Сохраняем изображение в буфер в формате PNG
buf = io.BytesIO()
img.save(buf, format="PNG")
mask_b64 = base64.b64encode(buf.getvalue()).decode()

# Случайный bbox и вероятность
bbox = [random.randint(0, 50) for _ in range(4)]
class_id = "whale"
prob = round(random.uniform(0.8, 1.0), 3)

# Результат
result = {
    "image_ind": filename,
    "bbox": bbox,
    "class_animal": class_id,
    "id_animal": "23",
    "probability": prob,
    "mask": mask_b64
}

import yaml
print(yaml.dump(result, allow_unicode=True))


In [ ]:
import random
from rembg import remove
from PIL import Image
import base64
import io

def generate_base64_mask_with_removed_background(img_bytes: bytes) -> str:
    """
    Принимает байты изображения, удаляет фон и возвращает base64 PNG без фона.
    """
    # Удаление фона
    output = remove(img_bytes)
    
    # Преобразование результата в PNG и кодирование в base64
    processed_image = Image.open(io.BytesIO(output)).convert("RGBA")
    buf = io.BytesIO()
    processed_image.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode(), processed_image


def detection_id(filename: str, img_bytes: bytes) -> dict:
    bbox = [random.randint(0, 50) for _ in range(4)]
    class_id = "whale"
    prob = round(random.uniform(0.8, 1.0), 3)

    img = Image.open(io.BytesIO(img_bytes)).convert("RGBA")
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    
    mask_b64, processed = generate_base64_mask_with_removed_background(img_bytes)

    return {
        "image_ind": filename,
        "bbox": bbox,
        "class_animal": class_id,
        "id_animal": "23",
        "probability": prob,
        "mask": mask_b64,
        'processed': processed
    }

In [ ]:
import yaml

filename = "000fe6ebfc9893.jpg"
filepath = "../../data/datasets/" + filename

# Чтение байтов изображения
with open(filepath, "rb") as f:
    img_bytes = f.read()

# Генерация base64-маски с удалённым фоном
mask_b64, processed = generate_base64_mask_with_removed_background(img_bytes)

# Генерация bbox и вероятности
bbox = [random.randint(0, 50) for _ in range(4)]
class_id = "whale"
prob = round(random.uniform(0.8, 1.0), 3)

# Результат
result = {
    "image_ind": filename,
    "bbox": bbox,
    "class_animal": class_id,
    "id_animal": "23",
    "probability": prob,
    "mask": mask_b64,
    'processed': processed
}

# Вывод в YAML
print(yaml.dump(result, allow_unicode=True))

In [ ]:
result['processed']

In [ ]:
import yaml

with open("../../whales_be_service/src/whales_be_service/config.yaml", "r") as f:
    config = yaml.safe_load(f)

unique_names = set(config["id_to_name"].values())
print(f"Уникальных видов: {len(unique_names)}")
print("Список уникальных видов:", unique_names)


In [ ]:
len(set(config['id_to_name'].keys()))

In [ ]:

# First pass: count occurrences
name_counts = {}
for name in config['id_to_name'].values():
    name_counts[name] = name_counts.get(name, 0) + 1

# Second pass: build new mapping with suffixes where necessary
new_mapping = {}
occ_tracker = {}
for key, name in config['id_to_name'].items():
    if name_counts[name] == 1:
        new_mapping[key] = name
    else:
        occ_tracker[name] = occ_tracker.get(name, 0) + 1
        new_mapping[key] = f"{name}_{occ_tracker[name]}"

# Prepare YAML output
new_yaml_content = yaml.safe_dump({'id_to_name': new_mapping}, allow_unicode=True, sort_keys=False)

# Save to file
output_path = 'config_suffix.yaml'
with open(output_path, 'w') as f_out:
    f_out.write(new_yaml_content)

In [ ]:
list(new_mapping.keys())[:10]

In [ ]:
import os
_LEGACY_VIT = '../demo-ui/models/model-e15.pt'
if not os.path.exists(_LEGACY_VIT):
    raise FileNotFoundError(
        'Legacy ViT checkpoint not downloaded. This cell uses the Stage-1 '
        'Vision Transformer L/32 model, which is NOT fetched by '
        'scripts/download_models.sh. Download manually from Yandex Disk '
        '(https://disk.yandex.ru/d/GshqU9o6nNz7ZA) and place at '
        '../demo-ui/models/model-e15.pt, or use the production '
        'EfficientNet-B4 checkpoint via whales_be_service.inference.get_pipeline().'
    )

import io, random, yaml, cv2, numpy as np, torch
from PIL import Image
from torch.utils.data import DataLoader
from albumentations.pytorch import ToTensorV2
import albumentations as A
import pandas as pd

# ---------- 1. модел��, трансформы, датасет ---------- #

CONFIG = {
    "img_size": 448,
    "test_batch_size": 1,
    "num_classes": 15_587,
    "patch_size": 32,
    "device": torch.device("cuda:0" if torch.cuda.is_available() else "cpu"),
}

data_transforms = A.Compose(
    [
        A.Resize(CONFIG["img_size"], CONFIG["img_size"]),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
        ),
        ToTensorV2(),
    ],
    p=1.0,
)


class CetaceanImageDataset(torch.utils.data.Dataset):
    def __init__(self, img_list, transforms=None):
        self.img_list = img_list
        self.transforms = transforms

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        img = self.img_list[idx]
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transforms:
            img = self.transforms(image=img)["image"]
        return {"image": img}


# ---------- 2. архитектура ---------- #

class AttentionBlock(torch.nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_heads, dropout=0.0):
        super().__init__()
        self.norm1 = torch.nn.LayerNorm(embed_dim)
        self.attn = torch.nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.norm2 = torch.nn.LayerNorm(embed_dim)
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(embed_dim, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim, embed_dim),
            torch.nn.Dropout(dropout),
        )

    def forward(self, x):
        y = self.norm1(x)
        x = x + self.attn(y, y, y)[0]
        x = x + self.mlp(self.norm2(x))
        return x


def img_to_patch(x, patch, flat=True):
    B, C, H, W = x.shape
    x = x.reshape(B, C, H // patch, patch, W // patch, patch)
    x = x.permute(0, 2, 4, 1, 3, 5).flatten(1, 2)
    if flat:
        x = x.flatten(2, 4)
    return x


class VisionTransformer(torch.nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_channels, num_heads, num_layers,
                 num_classes, patch_size, num_patches, dropout=0.0):
        super().__init__()
        self.patch_size = patch_size
        self.inp = torch.nn.Linear(num_channels * patch_size ** 2, embed_dim)
        self.blocks = torch.nn.Sequential(
            *[AttentionBlock(embed_dim, hidden_dim, num_heads, dropout) for _ in range(num_layers)]
        )
        self.mlp_head = torch.nn.Sequential(
            torch.nn.LayerNorm(embed_dim), torch.nn.Linear(embed_dim, num_classes)
        )
        self.cls_token = torch.nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_emb = torch.nn.Parameter(torch.randn(1, 1 + num_patches, embed_dim))
        self.drop = torch.nn.Dropout(dropout)

    def forward(self, x):
        x = img_to_patch(x, self.patch_size)
        B, T, _ = x.shape
        x = self.inp(x)
        cls = self.cls_token.repeat(B, 1, 1)
        x = torch.cat([cls, x], 1) + self.pos_emb[:, : T + 1]
        x = self.drop(x).transpose(0, 1)
        x = self.blocks(x)
        return self.mlp_head(x[0])


# ---------- 3. инициализация ---------- #

CSV_PATH = "../demo-ui/resources/database.csv"
df = pd.read_csv(CSV_PATH)
uniq_df = df.drop_duplicates("individual_id")
CLASS_ID_LIST = uniq_df["individual_id"].astype(str).tolist()
ID_TO_NAME = dict(zip(uniq_df["individual_id"].astype(str), uniq_df["species"]))

assert len(CLASS_ID_LIST) == 15_587, f"Ожидалось 15 587, а получили {len(CLASS_ID_LIST)}"

_model = VisionTransformer(
    embed_dim=784, hidden_dim=1568, num_heads=8, num_layers=6,
    patch_size=32, num_channels=3, num_patches=196,
    num_classes=len(CLASS_ID_LIST), dropout=0.2,
).to(CONFIG["device"]).eval()

ckpt = torch.load("../demo-ui/models/model-e15.pt", map_location=CONFIG["device"])
_model.load_state_dict(ckpt["model_state_dict"], strict=False)


# ---------- 4. утили��а удаления фона ---------- #

from rembg import remove as rembg_remove
def generate_base64_mask_with_removed_background(img_bytes: bytes) -> str:
    import base64
    buf = io.BytesIO(rembg_remove(img_bytes))
    encoded = base64.b64encode(buf.getvalue()).decode()
    return encoded


# ---------- 5. Публичная функция ---------- #

def detection_id(filename: str, img_bytes: bytes) -> dict:
    np_img = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)
    loader = DataLoader(
        CetaceanImageDataset([np_img], transforms=data_transforms),
        batch_size=1, shuffle=False,
    )

    with torch.no_grad():
        batch = next(iter(loader))["image"].to(CONFIG["device"])
        logits = _model(batch)
        probs = torch.softmax(logits, 1)
        top_prob, top_idx = probs[0].max(0)

    class_idx = int(top_idx.item())
    class_id = CLASS_ID_LIST[class_idx]
    name = ID_TO_NAME.get(class_id, class_id)

    bbox = [0, 0, np_img.shape[1], np_img.shape[0]]
    mask_b64 = generate_base64_mask_with_removed_background(img_bytes)

    return {
        "image_ind": filename,
        "bbox": bbox,
        "class_animal": class_id,
        "id_animal": name,
        "probability": round(float(top_prob), 4),
        "mask": mask_b64,
    }


# ---------- 6. быстрый тест ---------- #
filename = "000fe6ebfc9893.jpg"
test_path = "../../data/datasets/" + filename
with open(test_path, "rb") as f:
    res = detection_id(test_path, f.read())
import pprint; pprint.pprint(res)

In [ ]:
df

In [ ]:
# once_build_meta.py
import pandas as pd, json, gzip, pathlib
df = pd.read_csv("../demo-ui/resources/database.csv").drop_duplicates("individual_id")
meta = {
    "id_list" : df["individual_id"].astype(str).tolist(),
    "id2name" : dict(zip(df["individual_id"].astype(str), df["species"]))
}
pathlib.Path("whale_meta.json.gz").write_bytes(gzip.compress(json.dumps(meta).encode()))

In [ ]:
import json, gzip, pathlib
meta = json.loads(gzip.decompress(pathlib.Path("whale_meta.json.gz").read_bytes()))
ID_LIST = meta["id_list"]
ID2NAME = meta["id2name"]


In [ ]:
ID_LIST